# Currency Converter - Extended Solution

**Project:** Chapter 14 Currency Converter (Hassan S. *Learn Python by Doing*)  
**Extended with:** Offline rates (primary) + optional API fallback, multi-currency conversion, conversion history, dictionary-of-lambdas alternate, extra practice exercises, and parameterised Monte-Carlo simulation of rate noise with histograms.

---

## Overview
A practical currency converter that converts an amount from a source currency into one or more target currencies. Because the sandbox has no outbound internet, the primary path uses a fixed offline rate table (relative to USD). Live API calls are attempted only when explicitly requested and fall back gracefully.

## Key Concepts
- Exchange-rate table (offline dict) or live API
- Validation of ISO-like currency codes
- Arithmetic: `converted = amount * (rate_to / rate_from)`
- Multi-target conversion & history tracking
- Alternate pure-function / lambda dispatch

## Flowchart of the Desired Outcome
![Currency Converter Flowchart](currency_converter_flowchart.png)


## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Optional, Callable
from datetime import datetime
from copy import deepcopy

print("Libraries imported successfully.")


## 1. Offline Exchange-Rate Table & Rate Loader

Rates are expressed relative to **1 USD**.  
Any pair conversion is performed by going through USD:

```
amount_in_target = amount * (RATES[target] / RATES[source])
```

An optional live-API path is provided; it is never required for the notebook to run.


In [ ]:
# Fixed offline rates (approximate, reproducible)
RATES_USD: Dict[str, float] = {
    "USD": 1.00,
    "EUR": 0.92,
    "GBP": 0.79,
    "JPY": 149.50,
    "INR": 83.20,
    "CRC": 515.00,   # Costa Rican Colón
    "CAD": 1.36,
    "AUD": 1.52,
    "CHF": 0.88,
    "CNY": 7.25,
}

SUPPORTED = set(RATES_USD.keys())
_rate_cache: Dict[str, Dict[str, float]] = {}
_history: List[dict] = []

def get_exchange_rates(base_currency: str = "USD", use_api: bool = False) -> Dict[str, float]:
    """Return rates relative to *base_currency*. Offline by default."""
    base = base_currency.upper()
    if base in _rate_cache:
        return _rate_cache[base]

    rates: Dict[str, float] = {}
    if use_api:
        try:
            import requests
            url = f"https://api.exchangerate-api.com/v4/latest/{base}"
            resp = requests.get(url, timeout=4)
            resp.raise_for_status()
            data = resp.json()
            rates = {k: float(v) for k, v in data["rates"].items() if k in SUPPORTED}
            print(f"[API] Live rates for base={base} obtained.")
        except Exception as exc:
            print(f"[API] Unavailable ({type(exc).__name__}); using offline table.")

    if not rates:
        if base not in RATES_USD:
            raise ValueError(f"Unsupported base currency: {base}")
        base_to_usd = RATES_USD[base]
        rates = {cur: (RATES_USD[cur] / base_to_usd) for cur in RATES_USD}

    _rate_cache[base] = rates
    return rates

# Quick sanity check
print("Supported currencies:", ", ".join(sorted(SUPPORTED)))
print("Sample rates vs USD:", {k: RATES_USD[k] for k in ["EUR", "GBP", "CRC"]})
print("Rates relative to EUR (first 4):", dict(list(get_exchange_rates("EUR").items())[:4]))


## 2. Core Conversion Functions

- `convert` – single pair  
- `convert_multi` – one amount → many targets  
- `record_conversion` / `show_history` – simple audit trail


In [ ]:
def convert(amount: float, from_currency: str, to_currency: str,
            rates: Optional[Dict[str, float]] = None) -> float:
    """Convert *amount* from one currency to another."""
    from_c = from_currency.upper()
    to_c   = to_currency.upper()
    if rates is None:
        rates = get_exchange_rates(from_c)
    if from_c not in rates or to_c not in rates:
        raise ValueError(f"Invalid currency code(s): {from_c}, {to_c}")
    if amount < 0:
        raise ValueError("Amount must be non-negative")
    return amount * (rates[to_c] / rates[from_c])


def convert_multi(amount: float, from_currency: str,
                  targets: List[str]) -> Dict[str, float]:
    """Convert one amount into several target currencies."""
    rates = get_exchange_rates(from_currency)
    return {t.upper(): round(convert(amount, from_currency, t, rates), 2)
            for t in targets}


def record_conversion(amount: float, from_c: str, to_c: str, result: float) -> None:
    _history.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "amount": amount,
        "from": from_c.upper(),
        "to": to_c.upper(),
        "result": round(result, 2),
    })


def show_history() -> None:
    if not _history:
        print("No conversions recorded yet.")
        return
    print("\n--- Conversion History ---")
    for i, h in enumerate(_history, 1):
        print(f"{i:2d}. {h['amount']:>10.2f} {h['from']} → {h['result']:>10.2f} {h['to']}  ({h['timestamp']})")


# ---- Demo (hard-coded so the notebook is fully reproducible) ----
print("=" * 55)
print("DEMO – Single & Multi-currency conversions (offline)")
print("=" * 55)

# Single
amt, src, dst = 100.0, "USD", "EUR"
res = convert(amt, src, dst)
record_conversion(amt, src, dst, res)
print(f"{amt} {src} = {res:.2f} {dst}")

# Another classic pair from the book table
print(f"50 EUR → GBP = {convert(50, 'EUR', 'GBP'):.2f}")

# Multi
print("\n250 EUR converts to:")
multi = convert_multi(250, "EUR", ["USD", "GBP", "JPY", "CRC", "INR"])
for k, v in multi.items():
    print(f"  → {v:10.2f} {k}")
    record_conversion(250, "EUR", k, v)

show_history()


## 3. Alternate Implementation – Dictionary of Lambdas

Instead of a single `convert` function we build a dictionary whose keys are strings such as `"USD2EUR"` and whose values are zero-argument-bound lambdas. This demonstrates pure dictionary dispatch (no if-elif chain, no function call overhead for the pair lookup).


In [ ]:
def make_converter(rates: Dict[str, float]) -> Dict[str, Callable[[float], float]]:
    """Return a dict of lambda converters for every ordered pair."""
    funcs = {}
    for src in rates:
        for dst in rates:
            if src == dst:
                continue
            # late binding avoided by default-arg capture
            funcs[f"{src}2{dst}"] = (
                lambda amt, s=src, d=dst, r=rates: amt * (r[d] / r[s])
            )
    return funcs


rates_usd = get_exchange_rates("USD")
ALT_FUNCS = make_converter(rates_usd)

print("Number of pair converters generated:", len(ALT_FUNCS))
print("Example keys:", list(ALT_FUNCS)[:6], "...")

# Use the alternate path
print(f"\nALT  75 USD → JPY  = {ALT_FUNCS['USD2JPY'](75):.2f}")
print(f"ALT 200 CRC → USD  = {ALT_FUNCS['CRC2USD'](200):.2f}")
print(f"ALT  30 GBP → EUR  = {ALT_FUNCS['GBP2EUR'](30):.2f}")


## 4. More Practice Exercises

1. **Batch conversion table** – convert a list of amounts from one currency into another.  
2. **Reverse lookup** – given a target amount, recover the original amount.  
3. **Cross-rate consistency check** – verify that A→B→C ≈ A→C (triangular arbitrage test).  
4. **Pretty report** for a tourist converting a fixed budget into several local currencies.


In [ ]:
# Practice 1 – Batch
print("Practice 1 – Batch USD → EUR")
amounts = [10, 25, 50, 100, 250, 1000]
print(f"{'USD':>8} {'EUR':>10}")
for a in amounts:
    print(f"{a:8.2f} {convert(a, 'USD', 'EUR'):10.2f}")

# Practice 2 – Reverse
print("\nPractice 2 – How many USD do I need for exactly 500 EUR?")
needed = convert(500, "EUR", "USD")
print(f"  → {needed:.2f} USD")

# Practice 3 – Triangular consistency (USD → EUR → GBP vs USD → GBP)
print("\nPractice 3 – Cross-rate check")
via = convert(convert(100, "USD", "EUR"), "EUR", "GBP")
direct = convert(100, "USD", "GBP")
print(f"  100 USD → EUR → GBP = {via:.4f}")
print(f"  100 USD → GBP       = {direct:.4f}")
print(f"  absolute difference = {abs(via - direct):.6f}  (should be ~0)")

# Practice 4 – Tourist budget
print("\nPractice 4 – Tourist with 1 000 USD budget")
budget = convert_multi(1000, "USD", ["EUR", "GBP", "JPY", "CRC", "CAD"])
for cur, val in budget.items():
    print(f"  {val:10.2f} {cur}")


## 5. Simulation Section – Modify Parameters & Observe Results

Change any of the parameters below and re-run the cell to obtain different histograms and statistics:

| Parameter   | Meaning                                      | Default |
|-------------|----------------------------------------------|---------|
| `base_amount` | Amount expressed in the source currency    | 100     |
| `from_cur`    | Source currency                            | USD     |
| `targets`     | List of target currencies                  | EUR,GBP,JPY,CRC |
| `n_sims`      | Number of Monte-Carlo draws                | 800     |
| `noise_pct`   | Relative standard deviation of rate noise  | 0.03    |
| `seed`        | Random seed for reproducibility            | 42      |

The simulation multiplies each true rate by a random factor ~ N(1, noise_pct) (clipped) and records the resulting converted amounts.


In [ ]:
def run_currency_simulation(
    base_amount: float = 100.0,
    from_cur: str = "USD",
    targets: Optional[List[str]] = None,
    n_sims: int = 800,
    noise_pct: float = 0.03,
    seed: int = 42,
):
    """Monte-Carlo rate-noise simulation with four histograms."""
    if targets is None:
        targets = ["EUR", "GBP", "JPY", "CRC"]
    rng = np.random.default_rng(seed)
    rates = get_exchange_rates(from_cur)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    fig.suptitle(
        f"Currency Converter – Simulation\n"
        f"(base = {base_amount} {from_cur}, noise σ={noise_pct*100:.1f} %, n={n_sims})",
        fontsize=13, fontweight="bold",
    )
    colors = ["#1565C0", "#2E7D32", "#C62828", "#6A1B9A"]

    stats = {}
    for ax, tgt, col in zip(axes.flat, targets, colors):
        true_rate = rates[tgt] / rates[from_cur]
        noisy = rng.normal(true_rate, true_rate * noise_pct, n_sims)
        noisy = np.clip(noisy, true_rate * 0.7, true_rate * 1.3)
        converted = base_amount * noisy
        stats[tgt] = {
            "mean": converted.mean(),
            "std": converted.std(),
            "true": base_amount * true_rate,
        }
        ax.hist(converted, bins=35, color=col, alpha=0.75, edgecolor="white")
        ax.axvline(converted.mean(), color="black", ls="--", lw=1.4,
                   label=f"mean={converted.mean():.2f}")
        ax.axvline(base_amount * true_rate, color="orange", ls=":", lw=2,
                   label=f"true={base_amount*true_rate:.2f}")
        ax.set_title(f"{from_cur} → {tgt}")
        ax.set_xlabel(f"Amount ({tgt})")
        ax.set_ylabel("Frequency")
        ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\nSimulation summary statistics:")
    print(f"{'Target':<8} {'True':>10} {'Mean':>10} {'Std':>10}")
    for t, s in stats.items():
        print(f"{t:<8} {s['true']:10.2f} {s['mean']:10.2f} {s['std']:10.2f}")
    return stats


# Default run
print("Running default simulation (change parameters and re-execute for new results)...")
_ = run_currency_simulation()


### 5b. Sensitivity – effect of noise magnitude

Re-run with a larger noise percentage to see how the distributions widen.


In [ ]:
print("Higher-noise simulation (noise_pct = 0.08) …")
_ = run_currency_simulation(noise_pct=0.08, seed=7)


## 6. Key Takeaways (Solution)

- Offline rate tables keep the program usable without network access and make results fully reproducible.
- All pair conversions reduce to a single multiplication once rates share a common base (USD).
- Dictionary-of-lambdas is a clean alternate to an explicit `convert` function and scales automatically with the number of currencies.
- A short history list turns the converter into a tiny audit log – useful for later reporting or GUI display.
- Monte-Carlo noise around the published rates quickly visualises exchange-rate risk for a given budget.

---
*End of Currency Converter Extended Solution*
